# shrewd quickstart: no API key needed

This runs the whole shrewd workflow on real data in a couple of minutes: hand labels become a
locked test set, an LLM's answers become training data, and you get a small local model, a
report that scores it and the LLM against the human labels, and a routing threshold with a
guarantee.

It doesn't need an API key because the repo ships the teacher's answers for this data
(Fable 5.1, cached). `judge()` finds every answer in that cache and makes no calls.
Everything else (calibration, training, scoring, the threshold) runs here for real.

The data is the public SMS Spam Collection, with four questions asked about each message.
It's a demo of the workflow, not a spam filter to deploy: the messages are from 2011.

In [1]:
%%capture
!pip install "shrewd[teacher]" huggingface_hub pyarrow
!git clone --depth 1 https://github.com/sshah03/shrewd shrewd-repo

Download the public data (500 hand-labeled messages, a pool of 1,500 unlabeled ones, and
a 4,000-message holdout) and look at the questions. Only `unsolicited` has human labels in
this dataset; the other three are answered by the teacher alone.

In [2]:
import gzip
import os
import shutil
import sys
import warnings
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"             # keep the output readable
# add_seed() warns that three questions have no hand labels, which is expected here
warnings.filterwarnings("ignore", category=UserWarning)
REPO = Path("shrewd-repo")
!python {REPO}/examples/panels/fetch.py messages
sys.path.insert(0, str(REPO / "examples" / "panels"))
from panels import PANELS  # noqa: E402

spec = PANELS["messages"]
for key, question in spec["questions"].items():
    print(f"{key:12} {question.kind:6}  {question.instructions}")

messages: seed 500, pool 1500, holdout 3160 -> shrewd-repo/data/messages


unsolicited  noul    Is this an unsolicited bulk, automated or commercial message rather than a personal or expected one?
kind         choice  What kind of message is this?
asks_action  noul    Does the message ask the recipient to do something — click a link, call or text a number, reply, or pay?
risk         score   How much harm could come to the recipient from doing what the message asks?


## The five calls

This is the same workflow as the README. The only extra step is copying the cached teacher
answers into the project directory first.

In [3]:
import pandas as pd

from shrewd import Decisions, load

run = Path("runs/sms")
shutil.rmtree(run, ignore_errors=True)
d = Decisions(run, questions=spec["questions"], instructions=spec["instructions"],
              teacher="anthropic/claude-fable-5-1")

# the teacher's cached answers, so judge() below costs nothing
judged = REPO / "examples" / "panels" / "judged"
with gzip.open(judged / "messages.cache.db.gz", "rb") as src, open(run / "cache.db", "wb") as dst:
    shutil.copyfileobj(src, dst)

data = REPO / "data" / "messages"
seed = pd.read_csv(data / "seed.csv")               # text + the hand label for unsolicited
for key in spec["questions"]:
    if key not in seed:
        seed[key] = None                           # no human labels for the other three

d.add_seed(seed, test_frac=0.35)                   # 325 dev rows, 175 locked test rows
d.judge(pd.read_csv(data / "pool.csv"))            # 1,500 teacher answers, all from the cache
result = d.distill(features="tfidf", route_budget=0.02)

seed: 325 dev / 175 test documents, 4 questions


judged 1500 documents × 4 questions for $0.00 (one call per document)
274 (document, question) pairs where the teacher was unsure are in needs_review.csv; edit pool_judged.csv to overrule it
holding 450 judged documents out of training to set the routing threshold (budget 2.0%, 95% guarantee)
training tfidf student on 1375 documents, 4 heads over one featurization


calibrating on 1050 judged documents (cross-fit, no API calls)


## The report

The teacher and the student are both scored against the human labels on the locked test
set. The routing section says where the threshold landed and what it's worth: the local
answers it allows, checked against people.

In [4]:
print(result.report())

shrewd decisions: 4 questions, tfidf soft student
trained on 1375 documents · scored on the locked test set (175 documents)

question       type                ECE   Brier   resol  headline
unsolicited    noul    teacher   0.027   0.004   0.801  1.000 AUROC
               noul    student   0.002   0.009   0.801  0.999 AUROC

before calibration student ECE, averaged over questions: 0.112 → 0.002 after

routing: answer locally when every question's confidence is at least 0.942. At most 2%
         of all requests get a local answer the teacher disagrees with (95% bound, set
         on 450 held-out documents, 36.9% of them local). On the test set 36.0% are
         local, and against 63 hand-labeled answers those are wrong 0.0% of the time
         (the teacher 0.0% on the same documents).

findings: none, nothing looks off


## Where the threshold sends things

`dec.local(texts)` says which messages clear the threshold. On the 3,160-message holdout,
which played no part in training, look at what stays local and what goes to the teacher.

In [5]:
dec = load(run)
hold = pd.read_csv(data / "holdout.csv")
texts = hold["text"].astype(str).tolist()
local = dec.local(texts)
spam = (hold["unsolicited"] == "yes").to_numpy()

print(f"{local.mean():.0%} of {len(texts):,} holdout messages are answered locally")
print(f"  {local[~spam].mean():.0%} of ordinary messages, {local[spam].mean():.0%} of spam\n")
for label, pick in (("local  ", local), ("teacher", ~local)):
    for text in [t for t, ok in zip(texts, pick, strict=True) if ok][:4]:
        print(f"{label}  {text[:80]}")

37% of 3,160 holdout messages are answered locally
  42% of ordinary messages, 1% of spam

local    That means you got an A in epi, she.s fine. She.s here now.
local    Every King Was Once A Crying Baby And Every Great Building Was Once A Map.. Not 
local    He didn't see his shadow. We get an early spring yay
local    Yar lor wait 4 my mum 2 finish sch then have lunch lor... I whole morning stay a
teacher  Hi this is yijue... It's regarding the 3230 textbook it's intro to algorithms se
teacher  I got it before the new year cos yetunde said she wanted to surprise you with it
teacher  S:)but he had some luck.2 catches put down:)
teacher  and  picking them up from various points


Nearly every spam message goes to the teacher. The panel asks four questions, a message
is only as sure as its least sure answer, and on spam it's least sure about `kind`, `risk`
and `asks_action`. That's the guarantee doing its job: it answers locally only where it can
show it agrees with the teacher, which here is mostly ordinary personal chat.

## Try your own

`route=True` answers the messages that clear the threshold and gives `None` for the rest.
In production you'd pass `fallback=d.ask` instead, which sends those to the teacher (that
needs an API key). Edit the list and run it again.

In [6]:
import time

messages = [
    "Awesome, be there in a minute",
    "hey are we still on for dinner at 7?",
    "WINNER!! You have been selected for a £1000 prize. Call 09061701461 to claim now",
    "URGENT: your parcel is held. Pay the £2.99 fee at http://royal-mail-fee.co to release it",
]
for text, answers in zip(messages, dec.decide(messages, route=True), strict=True):
    if answers is None:
        print(f"-> teacher  {text}")
    else:
        print(f"   local    {text}")
        print(f"            unsolicited={answers['unsolicited'].noul:.2f}  "
              f"kind={answers['kind'].choice}  risk={answers['risk'].score:.1f}")

batch = texts[:2000]
t0 = time.perf_counter()
dec.decide(batch)
ms = (time.perf_counter() - t0) * 1000 / len(batch)
print(f"\n{ms:.3f} ms per message, all four questions at once")

   local    Awesome, be there in a minute
            unsolicited=0.00  kind=personal  risk=0.0
-> teacher  hey are we still on for dinner at 7?
-> teacher  WINNER!! You have been selected for a £1000 prize. Call 09061701461 to claim now
-> teacher  URGENT: your parcel is held. Pay the £2.99 fee at http://royal-mail-fee.co to release it



0.087 ms per message, all four questions at once


## Using your own data

- Write your own questions with `Choice`, `Noul` and `Score`, hand-label a few hundred
  documents with one column per question, and set an API key for the teacher
  (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or any model string litellm understands).
- `d.judge(df, dry_run=True)` prices the teacher calls before you spend anything.
- `d.optimize()` tunes the teacher's prompt on your dev split with GEPA before judging.
- The routing threshold needs about a thousand judged documents to answer much locally.

The [README](https://github.com/sshah03/shrewd) covers the rest, and
[BENCHMARKS.md](https://github.com/sshah03/shrewd/blob/main/BENCHMARKS.md) has what I
measured.